# R GIS Ecosystem Demo: Vector, Raster, Interpolation, and Interactive Maps

This notebook demonstrates a non-trivial geospatial workflow in R using several ecosystem packages:

- **Vector analysis**: `sf`, `lwgeom`, `s2`
- **Raster analysis**: `terra`, `stars`
- **Spatial interpolation**: `gstat`
- **Visualization**: `ggplot2`, `tmap`, `leaflet`, `mapview`

It intentionally uses package-provided sample data where possible (`sf::nc`, `datasets::volcano`, `sp::meuse`).

In [ ]:
# Keep startup light; load libraries close to each demo section.
cat('sf external libs:\n')
print(sf::sf_extSoftVersion())

## 1. Load and Inspect Sample Vector Data (`sf::nc`)

In [ ]:
suppressPackageStartupMessages(library(sf))

nc <- st_read(system.file('shape/nc.shp', package = 'sf'), quiet = TRUE)

cat('Rows:', nrow(nc), '\n')
cat('CRS:', st_crs(nc)$input, '\n')
print(head(st_drop_geometry(nc), 3))

In [ ]:
suppressPackageStartupMessages(library(ggplot2))

ggplot(nc) +
  geom_sf(aes(fill = SID74)) +
  scale_fill_viridis_c(option = 'C') +
  labs(
    title = 'North Carolina SIDS counts (1974)',
    fill = 'SID74'
  ) +
  theme_minimal()

## 2. CRS Transform + Geometry-Derived Metrics

In [ ]:
suppressPackageStartupMessages({
  library(sf)
  library(dplyr)
})

# Project to a metric CRS before computing area/perimeter.
nc_32119 <- st_transform(nc, 32119)

nc_metrics <- nc_32119 %>%
  mutate(
    area_km2 = as.numeric(st_area(geometry)) / 1e6,
    perimeter_km = as.numeric(sf::st_perimeter(geometry)) / 1000,
    sid74_rate_per_1000_births = SID74 / BIR74 * 1000
  )

summary(nc_metrics$sid74_rate_per_1000_births)

In [ ]:
suppressPackageStartupMessages(library(ggplot2))

ggplot(nc_metrics) +
  geom_sf(aes(fill = sid74_rate_per_1000_births), color = NA) +
  scale_fill_viridis_c(option = 'D') +
  labs(
    title = 'SID rate per 1000 births (1974)',
    fill = 'Rate'
  ) +
  theme_minimal()

## 3. Spatial Predicates and Nearest Neighbors (`s2` geometry engine)

In [ ]:
suppressPackageStartupMessages({
  library(sf)
  library(dplyr)
})

# County centroids and nearest-neighbor graph by great-circle distance.
cent <- st_centroid(st_geometry(nc))
idx_nn <- st_nearest_feature(cent, cent)

# Remove self matches and compute geodesic distance (meters).
self <- seq_len(length(cent))
idx_nn[idx_nn == self] <- ifelse(idx_nn[idx_nn == self] == 1, 2, 1)

dist_m <- as.numeric(st_distance(cent, cent[idx_nn], by_element = TRUE))

nn_tbl <- tibble::tibble(
  county = nc$NAME,
  nearest_county = nc$NAME[idx_nn],
  nearest_distance_km = dist_m / 1000
)

print(head(arrange(nn_tbl, nearest_distance_km), 10))

## 4. Raster Processing from Base R Sample Data (`datasets::volcano`)

In [ ]:
suppressPackageStartupMessages(library(terra))

# Convert matrix to SpatRaster and create terrain derivatives.
r <- rast(volcano)
ext(r) <- ext(0, ncol(volcano), 0, nrow(volcano))
crs(r) <- 'EPSG:3857'
names(r) <- 'elevation'

slope <- terrain(r, v = 'slope', unit = 'radians')
aspect <- terrain(r, v = 'aspect', unit = 'radians')
hill <- shade(slope, aspect)

par(mfrow = c(1, 3), mar = c(3, 3, 2, 5))
plot(r, main = 'Elevation (volcano)')
plot(slope, main = 'Slope (radians)')
plot(hill, main = 'Hillshade')
par(mfrow = c(1, 1))

## 5. Vector-Raster Integration: Zonal Summary with Synthetic Polygons

In [ ]:
suppressPackageStartupMessages({
  library(sf)
  library(terra)
  library(ggplot2)
})

# Build a regular 4x4 polygon grid over the raster and compute mean elevation per cell.
grid_r <- rast(ext = ext(r), ncols = 4, nrows = 4, crs = crs(r))
values(grid_r) <- seq_len(ncell(grid_r))
grid_polys <- st_as_sf(as.polygons(grid_r, na.rm = FALSE))
grid_polys$tile_id <- seq_len(nrow(grid_polys))

# exactextractr may not always be available in all R envs; use terra::extract as fallback.
grid_vect <- vect(grid_polys)
zonal <- terra::extract(r, grid_vect, fun = mean, na.rm = TRUE, ID = FALSE)
grid_polys$mean_elevation <- zonal$elevation

ggplot(grid_polys) +
  geom_sf(aes(fill = mean_elevation), color = 'grey30', linewidth = 0.2) +
  scale_fill_viridis_c(option = 'C') +
  labs(title = 'Mean elevation per tile', fill = 'Mean elev.') +
  theme_minimal()

## 6. Spatial Interpolation (`gstat`) with `sp::meuse`

In [ ]:
suppressPackageStartupMessages({
  library(sp)
  library(sf)
  library(gstat)
  library(ggplot2)
})

data(meuse, package = 'sp')
meuse_sf <- st_as_sf(meuse, coords = c('x', 'y'), crs = 28992)

# Build prediction grid from the point extent.
bb <- st_bbox(meuse_sf)
pred_grid <- st_make_grid(meuse_sf, cellsize = 100, what = 'centers') %>%
  st_as_sf()

idw_out <- gstat::idw(
  formula = zinc ~ 1,
  locations = meuse_sf,
  newdata = pred_grid,
  idp = 2.0
)

ggplot() +
  geom_sf(data = idw_out, aes(color = var1.pred), size = 0.7, alpha = 0.8) +
  geom_sf(data = meuse_sf, color = 'black', size = 0.4) +
  scale_color_viridis_c(option = 'B') +
  labs(
    title = 'IDW interpolation of zinc concentration (meuse)',
    color = 'Pred. zinc'
  ) +
  theme_minimal()

## 7. Convert Raster to `stars` and Compare Structures

In [ ]:
suppressPackageStartupMessages(library(stars))

s_volcano <- st_as_stars(r)
print(s_volcano)

plot(s_volcano, main = 'volcano as stars object')

## 8. Interactive Visualization with `tmap`, `leaflet`, and `mapview`

In [ ]:
suppressPackageStartupMessages(library(tmap))

# tmap static style
tmap_mode('plot')
tm_shape(nc_metrics) +
  tm_polygons(
    fill = 'sid74_rate_per_1000_births',
    fill.scale = tm_scale(values = 'viridis'),
    fill.legend = tm_legend(title = 'SID rate')
  ) +
  tm_title('tmap static choropleth')

In [ ]:
suppressPackageStartupMessages({
  library(sf)
  library(leaflet)
})

# leaflet interactive map
nc_wgs84 <- st_transform(nc_metrics, 4326)
pal <- colorNumeric('viridis', domain = nc_wgs84$sid74_rate_per_1000_births)

leaflet(nc_wgs84) %>%
  addProviderTiles('CartoDB.Positron') %>%
  addPolygons(
    fillColor = ~pal(sid74_rate_per_1000_births),
    fillOpacity = 0.8,
    color = '#2b2b2b',
    weight = 0.6,
    popup = ~paste0(NAME, ': ', round(sid74_rate_per_1000_births, 2))
  ) %>%
  addLegend(
    pal = pal,
    values = ~sid74_rate_per_1000_births,
    title = 'SID rate / 1000 births'
  )

In [ ]:
suppressPackageStartupMessages(library(mapview))

# mapview quick inspection (interactive html widget)
mapview(nc_wgs84, zcol = 'sid74_rate_per_1000_births', layer.name = 'SID rate')

## 9. Optional: Persist Derived Attributes as Parquet

In [ ]:
suppressPackageStartupMessages({
  library(sf)
  library(arrow)
})

# Persist non-geometry attributes in Parquet format.
# (Geometry can be persisted separately e.g., GeoPackage/GeoParquet with additional tooling.)
attrs <- st_drop_geometry(nc_metrics)
arrow::write_parquet(attrs, 'nc_metrics_attributes.parquet')
cat('Wrote', nrow(attrs), 'rows to nc_metrics_attributes.parquet\n')